In [99]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules
import datetime as dt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from prophet import Prophet
import statsmodels.api as sm

df1 = pd.read_csv(r"C:\\Users\\belzs\\Documents\\DATA ANALYST\\Bike Store\\brands.csv")

df1

,brand_id,brand_name
0,1,Electra
1,2,Haro
2,3,Heller
3,4,Pure Cycles
4,5,Ritchey
5,6,Strider
6,7,Sun Bicycles
7,8,Surly
8,9,Trek


In [79]:
df2 = pd.read_csv(r"C:\\Users\\belzs\\Documents\\DATA ANALYST\\Bike Store\\Products.csv")

df2.head()

,product_id,product_name,brand_id,category_id,model_year,list_price
0,1,Trek 820 - 2016,9,6,2016,379.99
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99
2,3,Surly Wednesday Frameset - 2016,8,6,2016,999.99
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99


In [80]:
df3 = df2.merge(df1, how = 'inner', on = 'brand_id')

df3.head()



,product_id,product_name,brand_id,category_id,model_year,list_price,brand_name
0,1,Trek 820 - 2016,9,6,2016,379.99,Trek
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99,Ritchey
2,3,Surly Wednesday Frameset - 2016,8,6,2016,999.99,Surly
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99,Trek
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99,Heller


In [81]:
df_cat = pd.read_csv(r"C:\\Users\\belzs\\Documents\\DATA ANALYST\\Bike Store\\categories.csv")

df_cat.head()

df5 = df3.merge(df_cat, how = 'inner', on = 'category_id')
df5.head()


,product_id,product_name,brand_id,category_id,model_year,list_price,brand_name,category_name
0,1,Trek 820 - 2016,9,6,2016,379.99,Trek,Mountain Bikes
1,2,Ritchey Timberwolf Frameset - 2016,5,6,2016,749.99,Ritchey,Mountain Bikes
2,3,Surly Wednesday Frameset - 2016,8,6,2016,999.99,Surly,Mountain Bikes
3,4,Trek Fuel EX 8 29 - 2016,9,6,2016,2899.99,Trek,Mountain Bikes
4,5,Heller Shagamaw Frame - 2016,3,6,2016,1320.99,Heller,Mountain Bikes


In [82]:
df6 = df2.merge(df1, how = 'inner', on ='brand_id')\
   .merge(df_cat, how = 'inner', on ='category_id')
df6 = df6.drop(['brand_id', 'category_id'], axis = 1)
df6.head(10)
df6['product_name'].unique()
hx = df6['product_name'].str.replace(r"\s*-\s*\d{4}","", regex = True)
hx.unique()
df6['product_name'] = hx
df6.head()

,product_id,product_name,model_year,list_price,brand_name,category_name
0,1,Trek 820,2016,379.99,Trek,Mountain Bikes
1,2,Ritchey Timberwolf Frameset,2016,749.99,Ritchey,Mountain Bikes
2,3,Surly Wednesday Frameset,2016,999.99,Surly,Mountain Bikes
3,4,Trek Fuel EX 8 29,2016,2899.99,Trek,Mountain Bikes
4,5,Heller Shagamaw Frame,2016,1320.99,Heller,Mountain Bikes


In [83]:
df_ord = pd.read_csv(r"C:\\Users\\belzs\\Documents\\DATA ANALYST\\Bike Store\\orders.csv")
df_ord.tail()

,order_id,customer_id,order_status,order_date,required_date,shipped_date,store_id,staff_id
1610,1611,6,3,2018-09-06,2018-09-06,NaN,2,7
1611,1612,3,3,2018-10-21,2018-10-21,NaN,1,3
1612,1613,1,3,2018-11-18,2018-11-18,NaN,2,6
1613,1614,135,3,2018-11-28,2018-11-28,NaN,3,8
1614,1615,136,3,2018-12-28,2018-12-28,NaN,3,8


In [84]:
df_stores = pd.read_csv(r"C:\\Users\\belzs\\Documents\\DATA ANALYST\\Bike Store\\stores.csv")
df_stores.isna().sum()

store_id      0
store_name    0
phone         0
email         0
street        0
city          0
state         0
zip_code      0
dtype: int64

In [85]:
df_staffs = pd.read_csv(r"C:\\Users\\belzs\\Documents\\DATA ANALYST\\Bike Store\\staffs.csv")
df_staffs.head()
df_staffs['name'] = df_staffs['first_name'] + " " + df_staffs['last_name']
cols = list(df_staffs.columns)
cols.remove('name')
idx = cols.index('staff_id') + 1
cols.insert(idx, 'name')
df_staffs = df_staffs[cols]

df_staffs = df_staffs.drop(['first_name', 'last_name'], axis= True)

df_staffs.columns = df_staffs.columns.str.replace('name' ,'staffs_name')

df_staffs.head()


,staff_id,staffs_name,email,phone,active,store_id,manager_id
0,1,Fabiola Jackson,fabiola.jackson@bikes.shop,(831) 555-5554,1,1,NaN
1,2,Mireya Copeland,mireya.copeland@bikes.shop,(831) 555-5555,1,1,1.0
2,3,Genna Serrano,genna.serrano@bikes.shop,(831) 555-5556,1,1,2.0
3,4,Virgie Wiggins,virgie.wiggins@bikes.shop,(831) 555-5557,1,1,2.0
4,5,Jannette David,jannette.david@bikes.shop,(516) 379-4444,1,2,1.0


In [86]:
df_cus = pd.read_csv(r"C:\\Users\\belzs\\Documents\\DATA ANALYST\\Bike Store\\customers.csv")
df_cus['name'] = df_cus['first_name'] + ' ' + df_cus['last_name']

cols = list(df_cus.columns)
cols.remove('name')
idx = cols.index('customer_id') + 1
cols.insert(idx, 'name')
df_cus = df_cus[cols]

df_cus = df_cus.drop(['first_name', 'last_name'], axis= 1)
df_cus = df_cus.fillna('No Number')
df_cus['street'] = df_cus['street'].str.replace(r"\s*\.\s*$", "", regex=True)

df_cus.columns = df_cus.columns.str.replace('name', 'customers_name')

df_cus.tail()

,customer_id,customers_name,phone,email,street,city,state,zip_code
1440,1441,Jamaal Morrison,No Number,jamaal.morrison@msn.com,796 SE. Nut Swamp St,Staten Island,NY,10301
1441,1442,Cassie Cline,No Number,cassie.cline@gmail.com,947 Lafayette Drive,Brooklyn,NY,11201
1442,1443,Lezlie Lamb,No Number,lezlie.lamb@gmail.com,401 Brandywine Street,Central Islip,NY,11722
1443,1444,Ivette Estes,No Number,ivette.estes@gmail.com,88 N. Canterbury Ave,Canandaigua,NY,14424
1444,1445,Ester Acevedo,No Number,ester.acevedo@gmail.com,671 Miles Court,San Lorenzo,CA,94580


In [87]:
dfx  = df_ord.merge(df_cus[['customer_id','customers_name','phone','email','city','state']], on= 'customer_id', how= 'inner')\
      .merge(df_stores[['store_id','store_name']], on= 'store_id', how= 'inner')\
      .merge(df_staffs[['staff_id','staffs_name']], on= 'staff_id', how= 'inner')

dfx = dfx.drop(['customer_id', 'store_id','staff_id'], axis= True)
dfx.head(10)

,order_id,order_status,order_date,required_date,shipped_date,customers_name,phone,email,city,state,store_name,staffs_name
0,1,4,2016-01-01,2016-01-03,2016-01-03,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Santa Cruz Bikes,Mireya Copeland
1,2,4,2016-01-01,2016-01-04,2016-01-03,Jaqueline Cummings,No Number,jaqueline.cummings@hotmail.com,Huntington Station,NY,Baldwin Bikes,Marcelene Boyer
2,3,4,2016-01-02,2016-01-05,2016-01-03,Joshua Robertson,No Number,joshua.robertson@gmail.com,Patchogue,NY,Baldwin Bikes,Venita Daniel
3,4,4,2016-01-03,2016-01-04,2016-01-05,Nova Hess,No Number,nova.hess@msn.com,Duarte,CA,Santa Cruz Bikes,Genna Serrano
4,5,4,2016-01-03,2016-01-06,2016-01-06,Arla Ellis,No Number,arla.ellis@yahoo.com,Utica,NY,Baldwin Bikes,Marcelene Boyer
5,6,4,2016-01-04,2016-01-07,2016-01-05,Sharyn Hopkins,No Number,sharyn.hopkins@hotmail.com,Baldwinsville,NY,Baldwin Bikes,Marcelene Boyer
6,7,4,2016-01-04,2016-01-07,2016-01-05,Laureen Paul,No Number,laureen.paul@yahoo.com,Bellmore,NY,Baldwin Bikes,Marcelene Boyer
7,8,4,2016-01-04,2016-01-05,2016-01-05,Leslie Higgins,No Number,leslie.higgins@hotmail.com,Saratoga Springs,NY,Baldwin Bikes,Venita Daniel
8,9,4,2016-01-05,2016-01-08,2016-01-08,Neil Mccall,No Number,neil.mccall@gmail.com,San Carlos,CA,Santa Cruz Bikes,Mireya Copeland
9,10,4,2016-01-05,2016-01-06,2016-01-06,Alane Munoz,(914) 706-7576,alane.munoz@gmail.com,Yonkers,NY,Baldwin Bikes,Marcelene Boyer


In [88]:
df_item = pd.read_csv(r"C:\\Users\\belzs\\Documents\\DATA ANALYST\\Bike Store\\order_items.csv")
df_item.head()

dfg = df_item.merge(dfx,how='inner', on='order_id')

dfg.duplicated()
dfg.isna().sum()



dfh = dfg.merge(df6[['product_id', 'product_name', 'brand_name', 'category_name']], on = 'product_id', how = 'inner')

move = ['quantity', 'list_price', 'discount']
cols = [col for col in dfh.columns if col not in move] + move
dfh = dfh[cols]
dfh = dfh.drop([ 'item_id', 'product_id'], axis= 1)


depan = ['order_id','customers_name', 'phone', 'email', 'city', 'state', 'product_name', 'brand_name', 'category_name','store_name','staffs_name']
cols = depan + [col for col in dfh.columns if col not in depan]
dfh = dfh[cols]
dfh.head()

dfh['total'] = (dfh['quantity'] * dfh['list_price']*(1 - dfh['discount'] )).round(2)

dfh.head()

,order_id,customers_name,phone,email,city,state,product_name,brand_name,category_name,store_name,staffs_name,order_status,order_date,required_date,shipped_date,quantity,list_price,discount,total
0,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Electra Townie Original 7D EQ - Women's,Electra,Cruisers Bicycles,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,1,599.99,0.20,479.99
1,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Trek Remedy 29 Carbon Frameset,Trek,Mountain Bikes,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,2,1799.99,0.07,3347.98
2,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Surly Straggler,Surly,Cyclocross Bicycles,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,2,1549.00,0.05,2943.10
3,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Electra Townie Original 7D EQ,Electra,Cruisers Bicycles,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,2,599.99,0.05,1139.98
4,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Trek Fuel EX 8 29,Trek,Mountain Bikes,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,1,2899.99,0.20,2319.99


In [89]:
dfh['order_date'] = pd.to_datetime(dfh['order_date'])
dfh['required_date'] = pd.to_datetime(dfh['required_date'])
dfh['shipped_date'] = pd.to_datetime(dfh['shipped_date'])
dfh.info()

dfh.head()


<class 'pandas.DataFrame'>
RangeIndex: 4722 entries, 0 to 4721
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        4722 non-null   int64         
 1   customers_name  4722 non-null   str           
 2   phone           4722 non-null   str           
 3   email           4722 non-null   str           
 4   city            4722 non-null   str           
 5   state           4722 non-null   str           
 6   product_name    4722 non-null   str           
 7   brand_name      4722 non-null   str           
 8   category_name   4722 non-null   str           
 9   store_name      4722 non-null   str           
 10  staffs_name     4722 non-null   str           
 11  order_status    4722 non-null   int64         
 12  order_date      4722 non-null   datetime64[us]
 13  required_date   4722 non-null   datetime64[us]
 14  shipped_date    4214 non-null   datetime64[us]
 15  quantity       

,order_id,customers_name,phone,email,city,state,product_name,brand_name,category_name,store_name,staffs_name,order_status,order_date,required_date,shipped_date,quantity,list_price,discount,total
0,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Electra Townie Original 7D EQ - Women's,Electra,Cruisers Bicycles,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,1,599.99,0.20,479.99
1,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Trek Remedy 29 Carbon Frameset,Trek,Mountain Bikes,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,2,1799.99,0.07,3347.98
2,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Surly Straggler,Surly,Cyclocross Bicycles,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,2,1549.00,0.05,2943.10
3,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Electra Townie Original 7D EQ,Electra,Cruisers Bicycles,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,2,599.99,0.05,1139.98
4,1,Johnathan Velazquez,No Number,johnathan.velazquez@hotmail.com,Pleasanton,CA,Trek Fuel EX 8 29,Trek,Mountain Bikes,Santa Cruz Bikes,Mireya Copeland,4,2016-01-01,2016-01-03,2016-01-03,1,2899.99,0.20,2319.99


In [90]:
#FP GROWTH

basket = (dfh.groupby(['order_id', 'product_name'])['quantity']
          .sum().unstack().reset_index().fillna(0).set_index('order_id'))

basket_sets = (basket>=1).astype(bool)

frequent_itemsets = fpgrowth(basket_sets, min_support=0.01 , use_colnames= True)

rules = association_rules(frequent_itemsets, metric='lift', min_threshold= 1)

rules = rules.sort_values('confidence', ascending = False)

rules_clean = rules.copy()
rules_clean['antecedents'] = rules_clean['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_clean['consequents'] = rules_clean['consequents'].apply(lambda x: ', '.join(list(x)))

final_report = rules_clean[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
final_report['support'] = (final_report['support']).round(4).astype(float)
final_report['confidence'] = (final_report['confidence']).round(4).astype(float)
final_report['lift'] = final_report['lift'].round(2)

print("\n ===   Bundling Product Strategy Bike Store === \n")
print(final_report.head(10).to_string(index=False))

final_report.to_csv('FP-GROWTH.csv', index = False)


 ===   Bundling Product Strategy Bike Store === 

                               antecedents                            consequents  support  confidence  lift
               Pure Cycles William 3-Speed            Electra Cruiser 1 (24-Inch)   0.0118      0.2436  2.10
               Ritchey Timberwolf Frameset            Electra Townie Original 21D   0.0111      0.2338  2.01
               Pure Cycles William 3-Speed          Electra Townie Original 7D EQ   0.0111      0.2308  2.07
                            Electra Moto 1            Electra Cruiser 1 (24-Inch)   0.0130      0.2308  1.99
Pure Cycles Western 3-Speed - Women's/2016            Electra Townie Original 21D   0.0118      0.2135  1.83
                     Heller Shagamaw Frame Electra Girl's Hawaii 1 (16-inch)/2016   0.0118      0.2088  2.00
    Electra Girl's Hawaii 1 (20-inch)/2016 Electra Girl's Hawaii 1 (16-inch)/2016   0.0124      0.2000  1.91
Pure Cycles Western 3-Speed - Women's/2016            Electra Cruiser 1 (24-I

In [92]:
#segmentation customers Kmeans
snapshot_date = dfh['order_date'].max() + dt.timedelta(days= 1)

rfm = dfh.groupby('customers_name').agg({
    'order_date' : lambda x: (snapshot_date - x.max()).days,
    'order_id' : 'count',
    'total' : 'sum'
})
rfm.columns= ['recency', 'frequency', 'monetary']
print(rfm.head())

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

kmeans = KMeans(n_clusters=3 , n_init= 10, random_state= 42)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)
print(rfm.groupby('cluster').mean())

cluster_names = {
    0 : 'Loyal',
    1 : 'At Risk',
    2 : 'New/Occasional'
}

rfm['segment_name'] = rfm['cluster'].map(cluster_names)

rfm['segment_name']

                recency  frequency  monetary
customers_name                              
Aaron Knapp        1043          2   5927.55
Abbey Pugh          995          2   4132.99
Abby Gamble         265          7  32802.99
Abram Copeland      572          5  24607.02
Adam Henderson      721          3   3695.47
            recency  frequency      monetary
cluster                                     
0        359.942708   6.135417  14832.914688
1        474.067200   2.854400   3954.195744
2        882.993620   2.807018   3779.613636


customers_name
Aaron Knapp        New/Occasional
Abbey Pugh         New/Occasional
Abby Gamble                 Loyal
Abram Copeland              Loyal
Adam Henderson     New/Occasional
                        ...      
Zona Cameron              At Risk
Zora Ford                 At Risk
Zoraida Patton     New/Occasional
Zulema Browning    New/Occasional
Zulema Clemons     New/Occasional
Name: segment_name, Length: 1444, dtype: str

In [100]:
forecast_data = dfh.groupby('order_date')['quantity'].sum().reset_index()
forecast_data.columns = ['ds', 'y']
model = Prophet(yearly_seasonality= True, daily_seasonality= False)
model.fit(forecast_data)

future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

print(forecast)
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv('SALES_FORECAST.csv', index=False)

19:18:44 - cmdstanpy - INFO - Chain [1] start processing
19:18:44 - cmdstanpy - INFO - Chain [1] done processing


            ds      trend  yhat_lower  yhat_upper  trend_lower  trend_upper  \
0   2016-01-01   8.351017   -2.542128   14.552308     8.351017     8.351017   
1   2016-01-02   8.353883   -1.688252   14.445291     8.353883     8.353883   
2   2016-01-03   8.356749   -1.212306   14.751413     8.356749     8.356749   
3   2016-01-04   8.359615    0.230603   16.032763     8.359615     8.359615   
4   2016-01-05   8.362481   -2.069287   14.343482     8.362481     8.362481   
..         ...        ...         ...         ...          ...          ...   
810 2019-03-24  11.918384    4.415675   21.361072    11.917664    11.919079   
811 2019-03-25  11.921425    5.437418   21.783975    11.920683    11.922129   
812 2019-03-26  11.924466    4.319550   20.760891    11.923709    11.925176   
813 2019-03-27  11.927507    4.162428   20.128038    11.926736    11.928217   
814 2019-03-28  11.930548    5.087484   21.128196    11.929766    11.931266   

     additive_terms  additive_terms_lower  additive

In [103]:
dfh_cat = dfh[dfh['category_name']=='Mountain Bikes']

price_data = dfh_cat.groupby('list_price')['quantity'].sum().reset_index()
x = sm.add_constant(price_data['list_price'])
model = sm.OLS(price_data['quantity'], x).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               quantity   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.032
Method:                 Least Squares   F-statistic:                    0.1106
Date:                Wed, 15 Apr 2026   Prob (F-statistic):              0.742
Time:                        19:28:28   Log-Likelihood:                -168.75
No. Observations:                  30   AIC:                             341.5
Df Residuals:                      28   BIC:                             344.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         64.5184     22.093      2.920      0.0